# Self Attention Mechanism with Trainable weights

In [1]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [2]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2 # Defined by me

In [3]:
torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.rand(d_in,d_out))
W_key = torch.nn.Parameter(torch.rand(d_in,d_out))
W_value = torch.nn.Parameter(torch.rand(d_in,d_out))

In [4]:
q_2 = x_2 @ W_query
q_2

tensor([0.4306, 1.4551], grad_fn=<SqueezeBackward4>)

In [5]:
keys = inputs @ W_key
values = inputs @ W_value

In [6]:
keys_2 = keys[1]
attn_score_22 = torch.dot(q_2, keys_2)
attn_score_22


tensor(1.8524, grad_fn=<DotBackward0>)

In [7]:
attn_scores_2 = q_2 @ keys.T

In [8]:
d_k = keys.shape[1]
attn_weights_2 = torch.softmax(attn_scores_2/d_k**0.5,dim=-1)
attn_weights_2, attn_weights_2.sum()

(tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
        grad_fn=<SoftmaxBackward0>),
 tensor(1., grad_fn=<SumBackward0>))

In [9]:
context_vec_2 = attn_weights_2 @ values
context_vec_2

tensor([0.3061, 0.8210], grad_fn=<SqueezeBackward4>)

### Creating Self Attention Class

In [10]:
import torch.nn as nn

class SelfAttentionClassV1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = torch.nn.Parameter(torch.rand(d_in,d_out))
        self.W_key = torch.nn.Parameter(torch.rand(d_in,d_out))
        self.W_value = torch.nn.Parameter(torch.rand(d_in,d_out))

    def forward(self, x):
        q = x @ W_query
        keys = x @ W_key
        val = x @ W_value

        attn_score = q @ keys.T
        attn_weights = torch.softmax(attn_score / d_k**0.5, dim = -1)
        context_vec = attn_weights @ val

        return context_vec


In [11]:
torch.manual_seed(123)
sa = SelfAttentionClassV1(d_in, d_out)
sa(inputs)

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)

In [12]:
import torch.nn as nn

class SelfAttentionClassV2(nn.Module):
    def __init__(self, d_in, d_out, kqv_bias=False):
        super().__init__()
        self.W_query = torch.nn.Linear(d_in,d_out,bias=kqv_bias)
        self.W_key = torch.nn.Linear(d_in,d_out,bias=kqv_bias)
        self.W_value = torch.nn.Linear(d_in,d_out,bias=kqv_bias)

    def forward(self, x):
        q = self.W_query(x)
        keys = self.W_key(x)
        val = self.W_value(x)

        attn_score = q @ keys.T
        attn_weights = torch.softmax(attn_score / d_k**0.5, dim = -1)
        context_vec = attn_weights @ val

        return context_vec


In [13]:
torch.manual_seed(123)
sa_2 = SelfAttentionClassV2(d_in,d_out)
sa_2(inputs)

tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)

## Implementing attention mask 

In [14]:
q = sa_2.W_query(inputs)
keys = sa_2.W_key(inputs)
val = sa_2.W_value(inputs)

attn_score = q @ keys.T
attn_weights = torch.softmax(attn_score / d_k**0.5,dim = -1)

In [15]:
context_len = attn_weights.shape[1]
mask = torch.tril(torch.ones(context_len,context_len))
masked_attn_weights = attn_weights * mask
masked_attn_weights

tensor([[0.1717, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1636, 0.1749, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1637, 0.1749, 0.1746, 0.0000, 0.0000, 0.0000],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.0000, 0.0000],
        [0.1667, 0.1722, 0.1721, 0.1618, 0.1633, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<MulBackward0>)

In [16]:
row_sums = masked_attn_weights.sum(dim=1, keepdim=True)
masked_attn_weights_normalized = masked_attn_weights / row_sums
masked_attn_weights_normalized

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
        [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
        [0.1994, 0.2060, 0.2058, 0.1935, 0.1953, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<DivBackward0>)

### Implementing Dropout mask

In [17]:
torch.manual_seed(123)

layer = torch.nn.Dropout(0.5)

layer(masked_attn_weights_normalized)


tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6380, 0.6816, 0.6804, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.5090, 0.5085, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4120, 0.0000, 0.3869, 0.0000, 0.0000],
        [0.0000, 0.3418, 0.3413, 0.3308, 0.3249, 0.0000]],
       grad_fn=<MulBackward0>)

#### Implementing casual self-attention class

In [18]:
batch = torch.stack((inputs, inputs), dim=0)
batch.shape

torch.Size([2, 6, 3])

In [19]:
mask = torch.triu(torch.ones(context_len, context_len), diagonal = 1)
masked = attn_score.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[0.3111,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.1655, 0.2602,   -inf,   -inf,   -inf,   -inf],
        [0.1667, 0.2602, 0.2577,   -inf,   -inf,   -inf],
        [0.0510, 0.1080, 0.1064, 0.0643,   -inf,   -inf],
        [0.1415, 0.1875, 0.1863, 0.0987, 0.1121,   -inf],
        [0.0476, 0.1192, 0.1171, 0.0731, 0.0477, 0.0966]],
       grad_fn=<MaskedFillBackward0>)


In [24]:
import torch.nn as nn

class CasualAttentionClass(nn.Module):
    def __init__(self, d_in, d_out,context_length, dropout, kqv_bias=False):
        super().__init__()
        self.W_query = torch.nn.Linear(d_in,d_out,bias=kqv_bias)
        self.W_key = torch.nn.Linear(d_in,d_out,bias=kqv_bias)
        self.W_value = torch.nn.Linear(d_in,d_out,bias=kqv_bias)
        self.dropout = torch.nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        q = self.W_query(x)
        keys = self.W_key(x)
        val = self.W_value(x)

        attn_score = q @ keys.transpose(1, 2)
        attn_score.masked_fill_ (self.mask.bool() [:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_score / keys.shape[-1]**0.5,dim=1)
        attn_weights = self.dropout(attn_weights)
        context_vec = attn_weights @ val

        return context_vec


In [25]:
torch.manual_seed(123)

context_length = batch.shape[1]
dropout = 0.0

# fix the constructor call: 3rd argument is dropout, not context_length
ca = CasualAttentionClass(d_in, d_out,context_length, dropout=dropout)
ca(batch)



tensor([[[-0.0844,  0.0414],
         [-0.2264, -0.0039],
         [-0.4163, -0.0564],
         [-0.5014, -0.1011],
         [-0.7754, -0.1867],
         [-1.1632, -0.3303]],

        [[-0.0844,  0.0414],
         [-0.2264, -0.0039],
         [-0.4163, -0.0564],
         [-0.5014, -0.1011],
         [-0.7754, -0.1867],
         [-1.1632, -0.3303]]], grad_fn=<UnsafeViewBackward0>)

### Implementing Multi-head attention from Single-head attention

In [31]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out,context_length,dropout=0.0,qkv_bias=False, num_heads=2):
        super().__init__()
        self.heads = nn.ModuleList([CasualAttentionClass(d_in, d_out,context_length,dropout,qkv_bias) for _ in range(num_heads)])

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads],dim=-1)
    

In [32]:
torch.manual_seed(123)
context_length = 6
d_in, d_out = 3, 2
ma = MultiHeadAttentionWrapper(d_in, d_out, context_length)
ma(batch)

tensor([[[-0.0844,  0.0414,  0.0766,  0.0171],
         [-0.2264, -0.0039,  0.2143,  0.1185],
         [-0.4163, -0.0564,  0.3878,  0.2453],
         [-0.5014, -0.1011,  0.4992,  0.3401],
         [-0.7754, -0.1867,  0.7387,  0.4868],
         [-1.1632, -0.3303,  1.1224,  0.8460]],

        [[-0.0844,  0.0414,  0.0766,  0.0171],
         [-0.2264, -0.0039,  0.2143,  0.1185],
         [-0.4163, -0.0564,  0.3878,  0.2453],
         [-0.5014, -0.1011,  0.4992,  0.3401],
         [-0.7754, -0.1867,  0.7387,  0.4868],
         [-1.1632, -0.3303,  1.1224,  0.8460]]], grad_fn=<CatBackward0>)